### 加载文档

In [3]:
from langchain_community.document_loaders import (
    TextLoader,
    UnstructuredMarkdownLoader,
    UnstructuredPDFLoader,
    UnstructuredWordDocumentLoader
)


def load_documents():
    """
    加载多种类型的文档，包括text、markdown、PDF和Word文档

    Returns:
        list: 包含所有加载文档的列表
    """
    # 加载文本文件
    text_documents = TextLoader(
        "../knowledge_base/sample.txt",
        encoding="utf8"
    ).load()

    # 加载Markdown文件
    md_documents = UnstructuredMarkdownLoader(
        "../knowledge_base/sample.md"
    ).load()

    # 加载PDF文件
    pdf_documents = UnstructuredPDFLoader(
        "../knowledge_base/sample.pdf",
        mode="elements",  # 元素模式
        strategy="hi_res",  # 高分辨率策略
        # strategy="fast",
        languages=["eng", "chi_sim"],  # 支持的语言：英文和简体中文
    ).load()

    # 加载Word文档
    word_documents = UnstructuredWordDocumentLoader(
        "../knowledge_base/sample.docx"
    ).load()

    # 返回所有文档的列表
    return text_documents + md_documents + pdf_documents + word_documents


documents = load_documents()
print(documents)

[Document(metadata={'source': '../knowledge_base/sample.txt'}, page_content='始计第一\n\n孙子曰：兵者，国之大事，死生之地，存亡之道，不可不察也。\n\n故经之以五事，校之以计，而索其情：一曰道，二曰天，三曰地，四曰将，五曰法。\n\n道者，令民与上同意，可与之死，可与之生，而不畏危也；天者，阴阳、寒暑、时制也；地者，远近、险易、广狭、死生也；将者，智、信、仁、勇、严也；法者，曲制、官道、主用也。凡此五者，将莫不闻，知之者胜，不知者不胜。\n\n故校之以计，而索其情，曰：主孰有道？将孰有能？天地孰得？法令孰行？兵众孰强？士卒孰练？赏罚孰明？吾以此知胜负矣。\n\n将听吾计，用之必胜，留之；将不听吾计，用之必败，去之。\n\n计利以听，乃为之势，以佐其外。势者，因利而制权也。\n\n兵者，诡道也。故能而示之不能，用而示之不用，近而示之远，远而示之近。利而诱之，乱而取之，实而备之，强而避之，怒而挠之，卑而骄之，佚而劳之，亲而离之，攻其无备，出其不意。此兵家之胜，不可先传也。\n\n夫未战而庙算胜者，得算多也；未战而庙算不胜者，得算少也。多算胜，少算不胜，而况无算乎！吾以此观之，胜负见矣。\n\n作战第二\n\n孙子曰：凡用兵之法，驰车千驷，革车千乘，带甲十万，千里馈粮。则内外之费，宾客之用，胶漆之材，车甲之奉，日费千金，然后十万之师举矣。\n\n其用战也，贵胜，久则钝兵挫锐，攻城则力屈，久暴师则国用不足。夫钝兵挫锐，屈力殚货，则诸侯乘其弊而起，虽有智者，不能善其后矣。故兵闻拙速，未睹巧之久也。夫兵久而国利者，未之有也。故不尽知用兵之害者，则不能尽知用兵之利也。\n\n善用兵者，役不再籍，粮不三载，取用于国，因粮于敌，故军食可足也。国之贫于师者远输，远输则百姓贫；近于师者贵卖，贵卖则百姓竭，财竭则急于丘役。力屈财殚，中原内虚于家，百姓之费，十去其七；公家之费，破军罢马，甲胄矢弩，戟楯矛橹，丘牛大车，十去其六。\n\n故智将务食于敌，食敌一钟，当吾二十钟；萁秆一石，当吾二十石。故杀敌者，怒也；取敌之利者，货也。故车战，得车十乘以上，赏其先得者，而更其旌旗。车杂而乘之，卒善而养之，是谓胜敌而益强。\n\n故兵贵胜，不贵久。故知兵之将，民之司命。国家安危之主也。\n\n谋攻第三\n\n孙子曰：凡用兵

### 数据清洗

In [6]:
import re
import json
def clean_content(documents: list):
    """文本清洗"""
    cleaned_docs = []
    for doc in documents:
        # 1、page_content处理：去除多余换行和空格
        text = doc.page_content
        #将连续的换行符替换为两个换行符，正则表达式模式：r"\n{2,}"
        # r""表示原始字符串（raw string），避免转义字符的特殊处理
        # \n表示换行符
        # {2,}是量词，表示前面的字符（换行符）出现2次或更多次
        text = re.sub(r"\n{2,}", "\n\n", text)
        text = text.strip()
        # 2、metadata处理：将所有非Chroma支持类型的值转为JSON格式字符串
        allowed_types = (str, int, float, bool)
        for key, value in doc.metadata.items():
            if not isinstance(value, allowed_types):
                try:
                    doc.metadata[key] = json.dumps(value,ensure_ascii=False)
                except (TypeError, ValueError):
                    #如果json.dumps失败（如包含不可序列化对象），转为str
                    doc.metadata[key] = str(value)
                    # 3、更新文档内容
                    doc.page_content = text
            cleaned_docs.append(doc)
    return cleaned_docs

### 数据切分

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#文本分块
text_splitter = RecursiveCharacterTextSplitter(
    #分隔符列表
    separators=["\n\n", "。"],
    #每个块的最大长度
    chunk_size=400,
    #每个块重叠的长度
    chunk_overlap=40,
)
documents = load_documents()
clean_documents = clean_content(documents)
print(clean_documents)
texts = text_splitter.split_documents(clean_documents)
print(texts)

[Document(metadata={'source': '../knowledge_base/sample.txt'}, page_content='始计第一\n\n孙子曰：兵者，国之大事，死生之地，存亡之道，不可不察也。\n\n故经之以五事，校之以计，而索其情：一曰道，二曰天，三曰地，四曰将，五曰法。\n\n道者，令民与上同意，可与之死，可与之生，而不畏危也；天者，阴阳、寒暑、时制也；地者，远近、险易、广狭、死生也；将者，智、信、仁、勇、严也；法者，曲制、官道、主用也。凡此五者，将莫不闻，知之者胜，不知者不胜。\n\n故校之以计，而索其情，曰：主孰有道？将孰有能？天地孰得？法令孰行？兵众孰强？士卒孰练？赏罚孰明？吾以此知胜负矣。\n\n将听吾计，用之必胜，留之；将不听吾计，用之必败，去之。\n\n计利以听，乃为之势，以佐其外。势者，因利而制权也。\n\n兵者，诡道也。故能而示之不能，用而示之不用，近而示之远，远而示之近。利而诱之，乱而取之，实而备之，强而避之，怒而挠之，卑而骄之，佚而劳之，亲而离之，攻其无备，出其不意。此兵家之胜，不可先传也。\n\n夫未战而庙算胜者，得算多也；未战而庙算不胜者，得算少也。多算胜，少算不胜，而况无算乎！吾以此观之，胜负见矣。\n\n作战第二\n\n孙子曰：凡用兵之法，驰车千驷，革车千乘，带甲十万，千里馈粮。则内外之费，宾客之用，胶漆之材，车甲之奉，日费千金，然后十万之师举矣。\n\n其用战也，贵胜，久则钝兵挫锐，攻城则力屈，久暴师则国用不足。夫钝兵挫锐，屈力殚货，则诸侯乘其弊而起，虽有智者，不能善其后矣。故兵闻拙速，未睹巧之久也。夫兵久而国利者，未之有也。故不尽知用兵之害者，则不能尽知用兵之利也。\n\n善用兵者，役不再籍，粮不三载，取用于国，因粮于敌，故军食可足也。国之贫于师者远输，远输则百姓贫；近于师者贵卖，贵卖则百姓竭，财竭则急于丘役。力屈财殚，中原内虚于家，百姓之费，十去其七；公家之费，破军罢马，甲胄矢弩，戟楯矛橹，丘牛大车，十去其六。\n\n故智将务食于敌，食敌一钟，当吾二十钟；萁秆一石，当吾二十石。故杀敌者，怒也；取敌之利者，货也。故车战，得车十乘以上，赏其先得者，而更其旌旗。车杂而乘之，卒善而养之，是谓胜敌而益强。\n\n故兵贵胜，不贵久。故知兵之将，民之司命。国家安危之主也。\n\n谋攻第三\n\n孙子曰：凡用兵

### 存入向量数据库

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
#加载嵌入模型
embedding_model =HuggingFaceEmbeddings(
model_name="../bge-base-zh-v1.5",
model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
encode_kwargs={"normalize_embeddings": True}, #输出归一化向量，更适合余弦相似度计算
)
#从Document中取出文本
page_content_list = [text.page_content for text in texts]
#进行嵌入
embeddings = embedding_model.embed_documents(page_content_list)
#打印嵌入结果
for i, (page_content, vector) in enumerate(zip(page_content_list,embeddings)):
    print("Text:\n", page_content)
    print("Embedding:\n",vector[:5])
    print()
    if i == 5:
        break

from langchain_chroma import Chroma
#嵌入并存储到向量数据库
vectorstore =Chroma.from_documents(texts,#文档列表
                                   embedding_model,#嵌入模型
                                   persist_directory="vectorstore",#存储路径
)

print(vectorstore.get().keys())#查看所有属性
print(vectorstore.get(include=["embeddings"])["embeddings"][:5, :5])#查看嵌入向量

In [10]:
from langchain_chroma import Chroma
vectorstore = Chroma(embedding_function=embedding_model,persist_directory="vectorstore",)
print(vectorstore.get().keys())#查看所有属性
print(vectorstore.get(include=["embeddings"])["embeddings"][:5, :5])#查看嵌入向量


dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])
[[-0.02093874 -0.02276396  0.02280377  0.015744   -0.01086405]
 [-0.0328072  -0.01315508  0.00030311  0.00980517  0.00491966]
 [-0.01325003 -0.02942497 -0.01212266  0.02137476  0.0064599 ]
 [-0.01234543 -0.01730887 -0.00748604  0.00202738  0.01503294]
 [-0.02486498 -0.0216117   0.00156077  0.02090647 -0.02317917]]
